In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

# Базовое разделение на признаки и целевую переменную
X = train.drop(['row_id', 'target'], axis=1)
y = train['target']
X_test = test.drop(['row_id'], axis=1)

# Сохраним ID для сабмита
test_ids = test['row_id']

# 1. FEATURE ENGINEERING
def create_features(df):
    df = df.copy()
    
    # Статусные признаки
    status_cols = [f'status_t{i}' for i in range(1, 7)]
    flow_cols = [f'flow_t{i}' for i in range(1, 7)]
    settle_cols = [f'settle_t{i}' for i in range(1, 7)]
    
    # 1.1. Агрегаты по временным рядам
    for cols, prefix in [(status_cols, 'status'), (flow_cols, 'flow'), (settle_cols, 'settle')]:
        # Базовые статистики
        df[f'{prefix}_mean'] = df[cols].mean(axis=1)
        df[f'{prefix}_std'] = df[cols].std(axis=1)
        df[f'{prefix}_median'] = df[cols].median(axis=1)
        df[f'{prefix}_min'] = df[cols].min(axis=1)
        df[f'{prefix}_max'] = df[cols].max(axis=1)
        df[f'{prefix}_last'] = df[cols[0]]
        df[f'{prefix}_first'] = df[cols[-1]]
        
        # Изменения во времени
        df[f'{prefix}_diff_last_first'] = df[cols[0]] - df[cols[-1]]
        df[f'{prefix}_trend'] = df.apply(lambda row: np.polyfit(range(6), row[cols], 1)[0], axis=1)
        
        # Последние изменения
        if len(cols) > 1:
            df[f'{prefix}_diff_last_2'] = df[cols[0]] - df[cols[1]]
    
    # 1.2. Взаимодействия между статусом, потоком и погашениями
    for i in range(1, 7):
        df[f'status_flow_ratio_t{i}'] = df[f'status_t{i}'] / (df[f'flow_t{i}'] + 1e-6)
        df[f'flow_settle_ratio_t{i}'] = df[f'flow_t{i}'] / (df[f'settle_t{i}'] + 1e-6)
        df[f'status_settle_diff_t{i}'] = df[f'status_t{i}'] - df[f'settle_t{i}']
    
    # 1.3. Агрегаты по циклам
    for i in range(1, 7):
        df[f'total_activity_t{i}'] = df[f'status_t{i}'] + df[f'flow_t{i}'] + df[f'settle_t{i}']
    
    # 1.4. Различия между соседними циклами
    for i in range(1, 6):
        for col_type in ['status', 'flow', 'settle']:
            col_current = f'{col_type}_t{i}'
            col_next = f'{col_type}_t{i+1}'
            df[f'{col_type}_diff_t{i}_t{i+1}'] = df[col_current] - df[col_next]
    
    # 1.5. Индикаторы изменений статуса
    status_diffs = [f'status_diff_t{i}_t{i+1}' for i in range(1, 6)]
    df['status_changes_count'] = df[status_diffs].apply(lambda x: (x != 0).sum(), axis=1)
    df['status_max_change'] = df[status_diffs].abs().max(axis=1)
    
    # 1.6. Взаимодействия с профильными признаками
    profile_features = ['cap_index', 'profile_code', 'segment_code', 'household_code', 'lifecycle_index']
    for pf in profile_features:
        for stat in ['status_last', 'flow_last', 'settle_last']:
            if stat in df.columns:
                df[f'{pf}_{stat}_interaction'] = df[pf] * df[stat]
    
    # 1.7. Пропущенные значения (если появятся после преобразований)
    df = df.fillna(df.median())
    
    return df

# Применяем feature engineering
X = create_features(X)
X_test = create_features(X_test)

# Выравниваем колонки (на случай если в тесте меньше фичей)
common_cols = list(set(X.columns) & set(X_test.columns))
X = X[common_cols]
X_test = X_test[common_cols]

# 2. МОДЕЛЬ LIGHTGBM
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_estimators': 1000,
    'verbose': -1
}

# 3. СТРАТИФИЦИРОВАННАЯ K-FOLD ВАЛИДАЦИЯ
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\nFold {fold + 1}/{n_folds}')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Обучение модели
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        early_stopping_rounds=100,
        verbose=False
    )
    
    # Предсказания
    val_preds = model.predict_proba(X_val)[:, 1]
    test_preds = model.predict_proba(X_test)[:, 1]
    
    oof_predictions[val_idx] = val_preds
    test_predictions += test_preds / n_folds
    
    # Оценка
    fold_score = roc_auc_score(y_val, val_preds)
    scores.append(fold_score)
    print(f'Fold {fold + 1} ROC-AUC: {fold_score:.5f}')

# 4. ОЦЕНКА МОДЕЛИ
print('\n' + '='*50)
print(f'Mean ROC-AUC: {np.mean(scores):.5f} (+/- {np.std(scores):.5f})')
print(f'OOF ROC-AUC: {roc_auc_score(y, oof_predictions):.5f}')

# 5. ФИНАЛЬНАЯ МОДЕЛЬ НА ВСЕХ ДАННЫХ
print('\nTraining final model on full data...')
final_model = lgb.LGBMClassifier(**params)
final_model.fit(X, y, verbose=False)

# 6. ПРЕДСКАЗАНИЯ ДЛЯ ТЕСТОВОГО НАБОРА
final_test_predictions = final_model.predict_proba(X_test)[:, 1]

# Используем усреднённые предсказания из CV
test_predictions = (test_predictions + final_test_predictions) / 2

# 7. СОЗДАНИЕ САБМИССИОННОГО ФАЙЛА
submission = pd.DataFrame({
    'row_id': test_ids,
    'target': test_predictions
})

# Проверяем формат
print(f'\nSubmission shape: {submission.shape}')
print(f'Target range: [{submission["target"].min():.4f}, {submission["target"].max():.4f}]')
print(f'Target mean: {submission["target"].mean():.4f}')

# Сохраняем
submission.to_csv('submission.csv', index=False)
print('\nSubmission saved to submission.csv')

# 8. ВАЖНЫЕ ПРИЗНАКИ
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nTop 20 important features:')
print(feature_importance.head(20).to_string(index=False))

# 9. ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ (опционально)
print('\n' + '='*50)
print('ADDITIONAL ANALYSIS:')
print(f'Target distribution in train: {y.value_counts(normalize=True).to_dict()}')
print(f'Number of features: {X.shape[1]}')

OSError: dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: '@rpath/libomp.dylib'
  Referenced from: '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib'
  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/lib/libomp.dylib' (no such file)